In [ ]:
import sys
import os
import subprocess
import numpy as np

# Add project root to path so `src_py` is importable as a package
_root = os.path.normpath(os.path.join(os.path.abspath(''), '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import matplotlib.pyplot as plt

from src_py import surrogate_model as srg, muvec as srg_grid
from src_py import surrogate_model_proj as srg_proj


In [ ]:
alpha = 4
gamma = 2.0
phi = 2.5
srg_vcut = srg(alpha, gamma, phi)
srg_proj_vcut = srg_proj(0.0, alpha, gamma, phi)

mu_max = 10
Nmu = 20
srg_proj_grid = np.linspace(0, mu_max, Nmu)

print(f"Surrogate model vcut: {srg_vcut}")
print(f"Surrogate projection vcut: {srg_proj_vcut}")

In [ ]:
figure, ax = plt.subplots(figsize=(5, 3.5))

vcut = srg_proj(srg_proj_grid, alpha, gamma, phi)
ax.plot(vcut, srg_proj_grid, marker='o', label='Surrogate Projection')
if srg_vcut is not None:
    ax.plot(srg_vcut, srg_grid, color='r', linestyle='--', label='Surrogate Model')
ax.axvline(x=np.sqrt(2*phi), color='k', linestyle=':', label=r'$\sqrt{2\phi}$')
ax.set_xlabel(r'$v_{\mathrm{cut}}$')
ax.set_ylabel(r'$\mu$')
ax.set_title(f'Surrogate Projection vs Model (α={alpha}, γ={gamma}, φ={phi})')
ax.legend()
plt.tight_layout()

In [ ]:
from src_py import generate_c_code

generate_c_code(
    nn_model      = "../model/nn_model.pth",
    svm_model     = "../model/svm_model.pkl",
    normalization = "../model/normalization.npz",
    output_dir    = "../generated_c_code",
    gkeyll_dir    = "/Users/ahoffman/gkeyll_main/gkeyll/gyrokinetic/ker/bc_sheath_gyrokinetic",
)

# --- Make clean and compile the C code ---
subprocess.run(["make", "-C", "../generated_c_code", "clean"]);
subprocess.run(["make", "-C", "../generated_c_code"]);

In [ ]:
# --- Run the compiled C test program and capture output ---
proc = subprocess.run(
    ["./../generated_c_code/test_surrogate", "predict"],
    capture_output=True, text=True
)
c_out = np.array([
    float(line.split("=")[1])
    for line in proc.stdout.strip().splitlines()
])

# --- Python reference (same inputs as test_surrogate.c: alpha=4, gamma=1.0, phi=2.5) ---
py_out = srg(4, 1.0, 2.5)

# --- Comparison ---
abs_err = np.abs(c_out - py_out)
print(f"{'i':>3}  {'C output':>14}  {'Python output':>14}  {'abs error':>12}")
print("-" * 50)
for i, (c, p, e) in enumerate(zip(c_out, py_out, abs_err)):
    print(f"{i:>3}  {c:>14.8f}  {p:>14.8f}  {e:>12.2e}")

print(f"\nMax absolute error : {abs_err.max():.2e}")
print(f"Mean absolute error: {abs_err.mean():.2e}")

In [ ]:
# -- Test eval mode ---
alpha = 5.7
gamma = 1.11
phi = 6.0
# alpha=5.729577951308232
# gamma=0.7407463044198136
# phi=3.999991596449837
mugrid_eval = np.linspace(0, 10, 32)

srg_vcut_eval = srg(alpha, gamma, phi)

c_vcut_eval = np.zeros_like(mugrid_eval)
for imu, mu in enumerate(mugrid_eval):
    proc_eval = subprocess.run(
        ["./../generated_c_code/test_surrogate", "eval", str(mu), str(alpha), str(gamma), str(phi)],
        capture_output=True, text=True
    )
    c_out = np.array([
        float(line.split("=")[1])
        for line in proc_eval.stdout.strip().splitlines()
    ])
    c_vcut_eval[imu] = c_out[0]

plt.figure(figsize=(5, 3.5))
plt.plot(c_vcut_eval, mugrid_eval, marker='o', label='C Eval')
if srg_vcut_eval is not None:
    plt.plot(srg_vcut_eval, srg_grid, color='r', linestyle='--', label='Python Eval')
plt.axvline(np.sqrt(phi), color='k', linestyle=':', label=r'$\sqrt{\phi}$')
plt.xlabel(r'$v_{\mathrm{cut}}$')
plt.ylabel(r'$\mu$')
plt.legend()

In [ ]:
# -- Test eval physical mode ---
Bimpact_angle = 0.1
Bmag = 1.0
dens_e = 1.2e19
Te = 5 * 1.60218e-19  # 10 eV in Joules
m_e = 9.10938356e-31
phi = 30.0 # V
phi_wall = 0.0 # V
epsilon_0 = 8.8541878128e-12
e = 1.602176634e-19

alpha = Bimpact_angle * 180 / np.pi
gamma = np.sqrt(m_e * dens_e / epsilon_0) / Bmag
phinorm = e*(phi - phi_wall) / Te

print(f"{alpha=}, {gamma=}, {phinorm=}")

mugrid_eval = np.linspace(0, 10 * Te / Bmag, 32)

srg_vals = np.zeros_like(mugrid_eval)
c_vals = np.zeros_like(mugrid_eval)

for imu, mu in enumerate(mugrid_eval):
    munorm = mu * Bmag / Te
    srg_vals[imu] = srg_proj(munorm, alpha, gamma, phinorm)

    proc_eval = subprocess.run(
        ["./../generated_c_code/test_surrogate", "physical", str(mu), str(phi), str(phi_wall), str(dens_e), str(Te), str(m_e), str(Bmag), str(Bimpact_angle)],
        capture_output=True, text=True
    )
    c_out = np.array([
        float(line.split("=")[1])
        for line in proc_eval.stdout.strip().splitlines()
    ])
    c_vals[imu] = c_out[0]

vcut_const = np.sqrt(e*(phi-phi_wall)/m_e)/(np.sqrt(Te/m_e))
# vcut_const = np.sqrt(phinorm)

plt.figure(figsize=(5, 3.5))
plt.plot(c_vals, mugrid_eval, marker='o', label='C Physical Eval')
plt.plot(srg_vals, mugrid_eval, color='r', linestyle='--', label='Python Physical Eval')
plt.axvline(vcut_const, color='k', linestyle=':', label=r'$\sqrt{\phi}$')
plt.xlabel(r'$v_{\mathrm{cut}}$')
plt.ylabel(r'$\mu$')
# plt.xlim(0, 2.2)
plt.legend()
plt.show()

In [ ]:
# -- Test eval physical vcut fact mode ---
Bimpact_angle = 0.1
Bmag = 1.5
dens_e = 1.2e19
Te = 5 * 1.60218e-19  # 10 eV in Joules
m_e = 9.10938356e-31
phi = 20.0 # V
phi_wall = 0.0 # V
epsilon_0 = 8.8541878128e-12
e = 1.602176634e-19

alpha = Bimpact_angle * 180 / np.pi
gamma = np.sqrt(m_e * dens_e / epsilon_0) / Bmag
phinorm = e*(phi - phi_wall) / Te

print(f"{alpha=}, {gamma=}, {phinorm=}")

mugrid_eval = np.linspace(0, 10 * Te / Bmag, 24)

srg_vals = np.zeros_like(mugrid_eval)
c_vals = np.zeros_like(mugrid_eval)
vcut_const = np.sqrt(phinorm)

for imu, mu in enumerate(mugrid_eval):
    munorm = mu * Bmag / Te
    srg_vals[imu] = srg_proj(munorm, alpha, gamma, phinorm)

    proc_eval = subprocess.run(
        ["./../generated_c_code/test_surrogate", "physical_vcut_fact", str(mu), str(phi), str(phi_wall), str(dens_e), str(Te), str(m_e), str(Bmag), str(Bimpact_angle)],
        capture_output=True, text=True
    )
    c_out = np.array([
        float(line.split("=")[1])
        for line in proc_eval.stdout.strip().splitlines()
    ])
    c_vals[imu] = c_out[0] * vcut_const  # Convert back to physical vcut

# vcut_const = np.sqrt(2*e*(phi-phi_wall)/m_e)/(np.sqrt(Te/m_e))


plt.figure(figsize=(5, 3.5))
plt.plot(c_vals, mugrid_eval, marker='o', label='C Physical Eval')
plt.plot(srg_vals, mugrid_eval, color='r', linestyle='--', label='Python Physical Eval')
plt.axvline(vcut_const, color='k', linestyle=':', label=r'$\sqrt{\phi}$')
plt.xlabel(r'$v_{\mathrm{cut}}$')
plt.ylabel(r'$\mu$')
plt.legend()
plt.show()